In [3]:
import sys
from pathlib import Path
import pandas as pd

print("Python:", sys.version.split()[0])
for pkg in ["pandas", "numpy", "scipy", "openpyxl", "matplotlib"]:
    try:
        m = __import__(pkg); print(f"{pkg:12s} OK       {m.__version__}")
    except ImportError:
        print(f"{pkg:12s} MISSING  -> run:  %pip install {pkg}")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("\nProject folder:", ROOT)

RAW = ROOT / "data" / "raw"
for folder in ["remitter", "beneficiary", "chargeback", "national", "kaggle"]:
    files = sorted(p.name for p in (RAW / folder).glob("*") if p.is_file())
    lock = [f for f in files if f.startswith("~$")]
    files = [f for f in files if not f.startswith("~$")]
    print(f"\n{folder}: {len(files)} files")
    for f in files:
        print("    ", f)
    if lock:
        print("     (ignored Excel lock files:", lock, "- close Excel to remove them)")

print("\n--- 1B: which national file is which? ---")
for f in sorted((RAW / "national").glob("*.xls*")):
    if f.name.startswith("~$"):
        continue
    d = pd.read_excel(f)
    cols = [str(c) for c in d.columns]
    if any("Avg" in c for c in cols):      kind = "VOLUME file"
    elif any("Banks" in c for c in cols):  kind = "BANKS file"
    else:                                  kind = "UNKNOWN"
    months = d.iloc[:, 0].dropna().astype(str).tolist()
    print(f"{f.name[:45]:45s} {kind:12s} rows={len(d):2d}  from {months[-1]} to {months[0]}")

Python: 3.13.5
pandas       OK       2.2.3
numpy        OK       2.3.5
scipy        OK       1.15.2
openpyxl     OK       3.1.5
matplotlib   OK       3.10.0

Project folder: C:\Users\rrron\OneDrive\Desktop\UPI_Failure_Analysis

remitter: 12 files
     2025-Dec-Remitter.xlsx
     2025-Nov-Remitter.xlsx
     2025-Oct-Remitter.xlsx
     2025-Sep-Remitter.xlsx
     2026-Apr-Remitter.xlsx
     2026-Aug-Remitter.xlsx
     2026-Feb-Remitter.xlsx
     2026-Jan-Remitter.xlsx
     2026-Jul-Remitter.xlsx
     2026-Jun-Remitter.xlsx
     2026-Mar-Remitter.xlsx
     2026-May-Remitter.xlsx

beneficiary: 12 files
     2025-Dec-Beneficiary.xlsx
     2025-Nov-Beneficiary.xlsx
     2025-Oct-Beneficiary.xlsx
     2025-Sep-Beneficiary.xlsx
     2026-Apr-Beneficiary.xlsx
     2026-Aug-Beneficiary.xlsx
     2026-Feb-Beneficiary.xlsx
     2026-Jan-Beneficiary.xlsx
     2026-Jul-Beneficiary.xlsx
     2026-Jun-Beneficiary.xlsx
     2026-Mar-Beneficiary.xlsx
     2026-May-Beneficiary.xlsx

chargeback: 11 files


In [4]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"

def num(x):
    """'29,52,542.05' -> 2952542.05  (NPCI stores numbers as text with commas)"""
    return float(re.sub(r"[,\s]", "", str(x)))

vol_parts, bank_parts = [], []
for f in sorted((RAW / "national").glob("*.xls*")):
    if f.name.startswith("~$"):
        continue
    d = pd.read_excel(f)
    cols = [str(c) for c in d.columns]
    if any("Avg" in c for c in cols):
        d.columns = ["month", "volume_mn", "avg_daily_volume_mn", "value_cr", "avg_daily_value_cr"]
        vol_parts.append(d)
    elif any("Banks" in c for c in cols):
        d.columns = ["month", "banks_live", "volume_mn", "value_cr"]
        bank_parts.append(d)
print("volume files:", len(vol_parts), "| banks files:", len(bank_parts))

vol = pd.concat(vol_parts).dropna(subset=["month"])
banks = pd.concat(bank_parts).dropna(subset=["month"])
for df in (vol, banks):
    for c in df.columns[1:]:
        df[c] = df[c].map(num)

m = vol.merge(banks, on="month", how="outer", suffixes=("", "_b"))
m["volume_mn"] = m["volume_mn"].fillna(m["volume_mn_b"])
m["value_cr"] = m["value_cr"].fillna(m["value_cr_b"])
m["month_start"] = pd.to_datetime(m["month"], format="%B-%Y")
m = m.sort_values("month_start").reset_index(drop=True)
m["avg_ticket_rs"] = m["value_cr"] * 1e7 / (m["volume_mn"] * 1e6)   # crore=1e7, million=1e6
m["is_partial_month"] = m["month_start"] >= "2026-09-01"            # unfinished current month

out = m[["month_start", "banks_live", "volume_mn", "value_cr", "avg_ticket_rs", "is_partial_month"]]
out.to_csv(CLEAN / "national_monthly.csv", index=False, float_format="%.2f")

print("\nrows:", len(out), "| first:", out.month_start.min().date(), "| last:", out.month_start.max().date())
print("duplicate months:", out.month_start.duplicated().sum(), "| missing volume:", out.volume_mn.isna().sum())
print(out.iloc[[0, 1, -2, -1]].to_string(index=False))

volume files: 3 | banks files: 3

rows: 30 | first: 2024-04-01 | last: 2026-09-01
duplicate months: 0 | missing volume: 0
month_start  banks_live  volume_mn   value_cr  avg_ticket_rs  is_partial_month
 2024-04-01       583.0   13303.99 1964464.52    1476.598013             False
 2024-05-01       598.0   14035.84 2044937.05    1456.939556             False
 2026-08-01       752.0   24508.95 2982355.93    1216.843614             False
 2026-09-01         NaN   22486.04 2727151.19    1212.819683              True


In [5]:
import re, hashlib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"
MON = {m: i for i, m in enumerate(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], 1)}

def clean_name(n):
    """'HDFC Bank Ltd.' -> 'HDFC BANK' (same key in remitter and beneficiary, so they can be joined later)"""
    n = re.sub(r"[.,]", " ", str(n).upper())
    n = re.sub(r"\b(LTD|LIMITED)\b", "", n)
    return re.sub(r"\s+", " ", n).strip()

def to_num(s):
    """'90.78%' -> 90.78 ; '5,833.74' -> 5833.74 ; '-' -> NaN"""
    return pd.to_numeric(s.astype(str).str.replace("%", "").str.replace(",", "").str.strip(), errors="coerce")

def load_top50(kind):                      # kind = "Remitter" or "Beneficiary"
    cols = ["sr_no", "bank_name", "volume_mn", "approved_pct", "bd_pct", "td_pct"]
    cols += ["reversal_count_mn", "reversal_success_pct"] if kind == "Remitter" else ["deemed_approved_pct"]
    frames, seen = [], {}
    for f in sorted((RAW / kind.lower()).glob("*.xlsx")):
        if f.name.startswith("~$"):
            continue
        y, mon = re.match(r"(\d{4})-([A-Za-z]{3})-", f.name).groups()
        month = f"{y}-{MON[mon]:02d}"                                   # month comes from the FILE NAME
        raw = pd.read_excel(f, header=None)
        # --- safety checks: did the right month get downloaded? ---
        digest = hashlib.md5(f.read_bytes()).hexdigest()
        if digest in seen:
            print(f"!! SAME CONTENT: {f.name} == {seen[digest]}  -> one of them is a wrong download")
        seen[digest] = f.name
        t = re.search(r"\(([A-Za-z]{3})[A-Za-z]*'(\d{2})\)", str(raw.iloc[0, 0]))
        if t and f"20{t.group(2)}-{MON[t.group(1).title()]:02d}" != month:
            print(f"?? TITLE MISMATCH: {f.name} says {t.group(0)} inside (Aug-2026 files say Jul'26: that one is NPCI's own mistake)")
        # --- read the table ---
        hdr = raw.index[raw[0].astype(str).str.contains("Sr")][0]       # header row = the one containing 'Sr. No.'
        d = raw.iloc[hdr + 1:, :len(cols)].copy()
        d.columns = cols
        d = d[pd.to_numeric(d["sr_no"], errors="coerce").notna()]       # drop footnote rows
        for c in cols[2:]:
            d[c] = to_num(d[c])
        d["month"] = month
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)                           # concat = stack all months one below the other
    df["bank_key"] = df["bank_name"].map(clean_name)
    # same bank listed twice in one month (Slice SFB): add volumes, volume-weight the percentages
    pct = ["approved_pct", "bd_pct", "td_pct"] + (["deemed_approved_pct"] if kind == "Beneficiary" else [])
    for c in pct:
        df[c] = df[c] * df["volume_mn"]
    agg = {"bank_name": "first", "volume_mn": "sum", **{c: "sum" for c in pct}}
    if kind == "Remitter":
        agg.update({"reversal_count_mn": "sum", "reversal_success_pct": "first"})
    df = df.groupby(["month", "bank_key"], as_index=False).agg(agg)
    for c in pct:
        df[c] = df[c] / df["volume_mn"]
    df["failure_pct"] = 100 - df["approved_pct"]
    return df.round(4)

remitter = load_top50("Remitter")
remitter.to_csv(CLEAN / "remitter_clean.csv", index=False)
print("\nREMITTER:", remitter.shape, "| months:", remitter.month.nunique(), "| banks:", remitter.bank_key.nunique())
w = remitter.groupby("month").apply(lambda g: pd.Series({"banks": len(g), "attempts_mn": g.volume_mn.sum(),
        "approved_%": (g.approved_pct * g.volume_mn).sum() / g.volume_mn.sum()}), include_groups=False)
print(w.round(2).to_string())

?? TITLE MISMATCH: 2026-Aug-Remitter.xlsx says (Jul'26) inside (Aug-2026 files say Jul'26: that one is NPCI's own mistake)

REMITTER: (598, 10) | months: 12 | banks: 59
         banks  attempts_mn  approved_%
month                                  
2025-09   50.0     20509.34       92.23
2025-10   50.0     21645.43       92.06
2025-11   50.0     21490.42       91.66
2025-12   50.0     22863.81       90.99
2026-01   50.0     22960.14       90.66
2026-02   50.0     21627.25       90.61
2026-03   49.0     24414.67       90.19
2026-04   50.0     24179.50       89.91
2026-05   50.0     25200.56       89.55
2026-06   49.0     24708.07       89.40
2026-07   50.0     25897.80       88.84
2026-08   50.0     26746.49       89.09


In [6]:
beneficiary = load_top50("Beneficiary")
beneficiary.to_csv(CLEAN / "beneficiary_clean.csv", index=False)
print("\nBENEFICIARY:", beneficiary.shape, "| months:", beneficiary.month.nunique(), "| banks:", beneficiary.bank_key.nunique())
w = beneficiary.groupby("month").apply(lambda g: pd.Series({"banks": len(g), "attempts_mn": g.volume_mn.sum(),
        "approved_%": (g.approved_pct * g.volume_mn).sum() / g.volume_mn.sum()}), include_groups=False)
print(w.round(2).to_string())

?? TITLE MISMATCH: 2026-Aug-Beneficiary.xlsx says (Jul'26) inside (Aug-2026 files say Jul'26: that one is NPCI's own mistake)

BENEFICIARY: (600, 9) | months: 12 | banks: 59
         banks  attempts_mn  approved_%
month                                  
2025-09   50.0     19449.07       99.50
2025-10   50.0     20510.31       99.57
2025-11   50.0     20289.51       99.50
2025-12   50.0     21439.27       99.47
2026-01   50.0     21535.34       99.39
2026-02   50.0     20218.62       99.45
2026-03   50.0     22428.03       99.48
2026-04   50.0     22116.14       99.57
2026-05   50.0     22973.47       99.50
2026-06   50.0     22491.11       99.53
2026-07   50.0     23429.53       99.48
2026-08   50.0     24253.46       99.55


In [7]:
def std(col):
    """Column names differ between months, so map them BY NAME, never by position."""
    c = re.sub(r"[^a-z]+", " ", str(col).lower()).strip()
    if c == "code": return "bank_code"
    if "beneficiary" in c: return "bank_name"
    if c.startswith("total txns"): return "total_txns"
    if "received" in c: return "cb_received"
    if "accepted" in c: return "cb_accepted"
    if "presentment" in c: return "represented"
    return c

frames = []
for f in sorted((RAW / "chargeback").glob("*.xlsx")):
    if f.name.startswith("~$"):
        continue
    y, mon = re.match(r"Chargeback-(\d{4})-([A-Za-z]{3})", f.name).groups()
    d = pd.read_excel(f).rename(columns=std)
    d = d[["bank_code", "bank_name", "total_txns", "cb_received", "cb_accepted", "represented"]].copy()
    for c in ["total_txns", "cb_received", "cb_accepted", "represented"]:
        d[c] = pd.to_numeric(d[c].astype(str).str.replace(",", ""), errors="coerce")
    d["month"] = f"{y}-{MON[mon]:02d}"
    frames.append(d)

cb = pd.concat(frames, ignore_index=True)
cb["bank_name"] = cb["bank_name"].astype(str).str.upper().str.strip()
cb["cb_ratio_pct"] = cb["cb_received"] / cb["total_txns"].where(cb["total_txns"] > 0) * 100       # recomputed
cb["cb_accept_rate_pct"] = cb["cb_accepted"] / cb["cb_received"].where(cb["cb_received"] > 0) * 100
cb["is_tiny_bank"] = cb["total_txns"] < 10000
cb.to_csv(CLEAN / "chargeback_clean.csv", index=False)

print("CHARGEBACK:", cb.shape, "| months:", cb.month.nunique())
print(cb.groupby("month").agg(entities=("bank_code", "nunique"), txns_mn=("total_txns", lambda s: s.sum() / 1e6),
                              chargebacks=("cb_received", "sum")).round(0).to_string())

CHARGEBACK: (3930, 10) | months: 11
         entities  txns_mn  chargebacks
month                                  
2025-09       793  21684.0       165936
2025-10       287  22836.0       161002
2025-11       298  20435.0       161854
2025-12       309  21602.0       181730
2026-01       326  21671.0       185903
2026-02       318  20372.0       168952
2026-03       324  22913.0       181908
2026-04       312  22607.0       210344
2026-05       323  23475.0       188907
2026-06       326  22975.0       193723
2026-07       313  23956.0       199441


In [8]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "clean"

festivals = """date,event_name,intensity,confidence
2025-09-22,Navratri begins,1,high
2025-09-23,Online mega-sale season starts,1,approx
2025-10-02,Dussehra,1,high
2025-10-10,Karwa Chauth,1,high
2025-10-18,Dhanteras,2,high
2025-10-20,Diwali,2,high
2025-10-21,Diwali (day 2),2,approx
2025-10-22,Govardhan Puja,1,approx
2025-10-23,Bhai Dooj,1,high
2025-11-05,Guru Nanak Jayanti,1,high
2025-12-25,Christmas,1,high
2025-12-31,New Year's Eve,2,high
2026-01-01,New Year's Day,1,high
2026-01-14,Makar Sankranti / Pongal,1,high
2026-01-26,Republic Day,1,high
2026-03-04,Holi,2,approx
2026-03-21,Eid-ul-Fitr,2,approx
2026-03-26,Ram Navami,1,approx
2026-05-28,Eid-ul-Adha (Bakrid),1,approx
2026-08-15,Independence Day,1,high
2026-08-28,Raksha Bandhan,1,approx
"""
(ROOT / "data" / "reference").mkdir(exist_ok=True)
(ROOT / "data" / "reference" / "festival_calendar.csv").write_text(festivals)

cal = pd.read_csv(ROOT / "data" / "reference" / "festival_calendar.csv", parse_dates=["date"])
d = pd.DataFrame({"date": pd.date_range("2025-09-01", "2026-08-31")})
d["month"] = d.date.dt.strftime("%Y-%m")
d["day_of_month"] = d.date.dt.day
d["weekday"] = d.date.dt.day_name()
d["weekday_no"] = d.date.dt.dayofweek                                              # Monday = 0
d["is_weekend"] = (d.weekday_no >= 5).astype(int)
d["is_month_end"] = (d.day_of_month >= d.date.dt.days_in_month - 1).astype(int)    # last 2 days of the month
d["is_month_start"] = (d.day_of_month <= 2).astype(int)
d = d.merge(cal[["date", "event_name", "intensity"]], on="date", how="left")
d["intensity"] = d.intensity.fillna(0).astype(int)                                 # 0 normal, 1 minor, 2 major festival
d["festival_name"] = d.event_name.fillna("")
d["is_festival"] = (d.intensity > 0).astype(int)
d["date"] = d.date.dt.strftime("%Y-%m-%d")
d = d.drop(columns="event_name")
d.to_csv(CLEAN / "dim_date.csv", index=False)

print("dim_date rows:", len(d), "| festival days:", d.is_festival.sum(), "| month-end days:", d.is_month_end.sum())
print(d[d.is_festival == 1][["date", "weekday", "festival_name", "intensity"]].head(6).to_string(index=False))

dim_date rows: 365 | festival days: 21 | month-end days: 24
      date  weekday                  festival_name  intensity
2025-09-22   Monday                Navratri begins          1
2025-09-23  Tuesday Online mega-sale season starts          1
2025-10-02 Thursday                       Dussehra          1
2025-10-10   Friday                   Karwa Chauth          1
2025-10-18 Saturday                      Dhanteras          2
2025-10-20   Monday                         Diwali          2


In [9]:
rem = pd.read_csv(CLEAN / "remitter_clean.csv")
ben = pd.read_csv(CLEAN / "beneficiary_clean.csv")
keys = sorted(set(rem.bank_key) | set(ben.bank_key))

PUBLIC = ["STATE BANK OF INDIA", "BANK OF BARODA", "PUNJAB NATIONAL BANK", "CANARA BANK", "UNION BANK OF INDIA", "BANK OF INDIA",
          "INDIAN BANK", "CENTRAL BANK OF INDIA", "INDIAN OVERSEAS BANK", "UCO BANK", "BANK OF MAHARASHTRA", "PUNJAB AND SIND BANK"]
PRIVATE = ["HDFC BANK", "ICICI BANK", "AXIS BANK", "KOTAK MAHINDRA BANK", "INDUSIND BANK", "YES BANK", "IDFC BANK", "FEDERAL BANK",
           "RBL BANK", "BANDHAN BANK", "CITY UNION BANK", "KARUR VYSYA BANK", "KARNATAKA BANK", "THE SOUTH INDIAN BANK",
           "TAMILNAD MERCANTILE BANK", "THE JAMMU & KASHMIR BANK", "IDBI BANK"]
PAYMENTS = ["AIRTEL PAYMENTS BANK", "FINO PAYMENTS BANK", "INDIA POST PAYMENTS BANK", "JIO PAYMENTS BANK", "NSDL PAYMENTS BANK"]
FOREIGN = ["CITI BANK", "CITIBANK CORPORATE", "DBS BANK INDIA", "STANDARD CHARTERED BANK"]

def bank_type(k):
    if k in PUBLIC: return "Public sector bank"
    if k in PRIVATE: return "Private sector bank"
    if "SMALL FINANCE" in k: return "Small finance bank"
    if k in PAYMENTS: return "Payments bank"
    if "GRAMIN" in k or "GRAMEENA" in k or k == "BARODA U P BANK": return "Regional rural bank"
    if k in FOREIGN: return "Foreign bank"
    if "CREDIT CARD" in k or "CARDS" in k: return "Credit-card UPI entity"
    return "CHECK MANUALLY"

dim_bank = pd.DataFrame({"bank_key": keys})
dim_bank["bank_type"] = dim_bank.bank_key.map(bank_type)
dim_bank.to_csv(CLEAN / "dim_bank.csv", index=False)

print("dim_bank rows:", len(dim_bank))
print(dim_bank.bank_type.value_counts().to_string())
print("\nFix these by hand (open dim_bank.csv in Excel):")
print(dim_bank[dim_bank.bank_type == "CHECK MANUALLY"].bank_key.tolist())

dim_bank rows: 66
bank_type
Private sector bank       17
Regional rural bank       13
Public sector bank        12
Small finance bank         9
Payments bank              5
Foreign bank               4
Credit-card UPI entity     3
CHECK MANUALLY             3

Fix these by hand (open dim_bank.csv in Excel):
['ONE MOBIKWIK SYSTEMS', 'SARASWAT CO-OPERATIVE BANK', 'TRI O TECH SOLUTIONS PRIVATE']


In [10]:
from scipy import stats

k = pd.read_csv(RAW / "kaggle" / "upi_transactions_2024.csv")
k.columns = ["txn_id", "ts", "txn_type", "category", "amount", "status", "sender_age", "receiver_age", "state",
             "sender_bank", "receiver_bank", "device", "network", "fraud", "hour", "dow", "is_weekend"]
k["failed"] = (k.status == "FAILED").astype(int)
k["amount_bucket"] = pd.cut(k.amount, [0, 100, 500, 1000, 5000, 10000, 1e9], right=False).astype(str)
print("rows:", len(k), "| overall failure rate: %.2f%%  (real NPCI is ~10%%)" % (k.failed.mean() * 100))

rows = []
for col in ["hour", "dow", "txn_type", "category", "sender_bank", "state", "device", "network", "sender_age", "amount_bucket"]:
    table = pd.crosstab(k[col], k.failed)                 # counts of success / failed in each group
    chi2, p, dof, expected = stats.chi2_contingency(table)
    rate = k.groupby(col).failed.mean() * 100
    rows.append([col, round(rate.min(), 2), round(rate.max(), 2), round(p, 3), p < 0.05])
print(pd.DataFrame(rows, columns=["factor", "lowest_fail_%", "highest_fail_%", "p_value", "significant?"]).to_string(index=False))

rows: 250000 | overall failure rate: 4.95%  (real NPCI is ~10%)
       factor  lowest_fail_%  highest_fail_%  p_value  significant?
         hour           4.54            5.40    0.886         False
          dow           4.77            5.10    0.277         False
     txn_type           4.88            5.09    0.817         False
     category           4.76            5.25    0.667         False
  sender_bank           4.82            5.10    0.862         False
        state           4.71            5.22    0.202         False
       device           4.93            5.15    0.554         False
      network           4.86            5.22    0.278         False
   sender_age           4.84            5.13    0.117         False
amount_bucket           4.76            5.76    0.163         False
